In [ ]:
import os
import os.path as osp
import json
import numpy as np
import torch
from PIL import Image
from shutil import copy2
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from torchvision.ops import nms

########################################
# Device Setup
########################################
device = "cuda:0" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

########################################
# Utility Functions (Data extraction & projection)
########################################

def get_sample_data(nusc, sample_token, sensor_name):
    sample = nusc.get('sample', sample_token)
    sd_token = sample['data'][sensor_name]
    sd = nusc.get('sample_data', sd_token)
    img_path = osp.join(nusc.dataroot, sd['filename'])
    img = Image.open(img_path).convert("RGB")
    return img, sd

def get_intrinsics(nusc, sample_data):
    cs = nusc.get('calibrated_sensor', sample_data['calibrated_sensor_token'])
    return np.array(cs['camera_intrinsic']).reshape(3, 3)

def project_3d_box_to_2d(nusc, sample_data, ann_token):
    box_3d = nusc.get_box(ann_token)
    ego_pose = nusc.get('ego_pose', sample_data['ego_pose_token'])
    box_3d.translate(-np.array(ego_pose['translation']))
    box_3d.rotate(Quaternion(ego_pose['rotation']).inverse)
    cs = nusc.get('calibrated_sensor', sample_data['calibrated_sensor_token'])
    box_3d.translate(-np.array(cs['translation']))
    box_3d.rotate(Quaternion(cs['rotation']).inverse)
    corners = box_3d.corners()  # shape (3,8)
    K = get_intrinsics(nusc, sample_data)
    uv_list = []
    for i in range(corners.shape[1]):
        x, y, z = corners[:, i]
        if z <= 0:
            uv_list.append(None)
            continue
        u = (K[0, 0] * x + K[0, 2] * z) / z
        v = (K[1, 1] * y + K[1, 2] * z) / z
        uv_list.append((u, v))
    valid_pts = [pt for pt in uv_list if pt is not None]
    if not valid_pts:
        return None
    us = [pt[0] for pt in valid_pts]
    vs = [pt[1] for pt in valid_pts]
    u_min, u_max = min(us), max(us)
    v_min, v_max = min(vs), max(vs)
    if u_max <= u_min or v_max <= v_min:
        return None
    return (u_min, v_min, u_max, v_max)

def clip_box_to_image(bbox, W, H):
    if bbox is None:
        return None
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(x1, W))
    x2 = max(0, min(x2, W))
    y1 = max(0, min(y1, H))
    y2 = max(0, min(y2, H))
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2, y2)

def crop_bounds(K, angle_min_deg, angle_max_deg, W):
    fx = K[0, 0]
    cx = K[0, 2]
    u_min = int(cx + fx * np.tan(np.deg2rad(angle_min_deg)))
    u_max = int(cx + fx * np.tan(np.deg2rad(angle_max_deg)))
    return max(0, min(u_min, W)), max(0, min(u_max, W))

def horizontal_overlap_ratio(bbox, u_min_crop, u_max_crop):
    if bbox is None:
        return 0.0
    x1, _, x2, _ = bbox
    inter_left = max(x1, u_min_crop)
    inter_right = min(x2, u_max_crop)
    overlap_w = max(0, inter_right - inter_left)
    total_w = x2 - x1
    if total_w <= 0:
        return 0.0
    return overlap_w / total_w

########################################
# (A) Generate COCO GT with Bounding Box Completeness Score (BCS)
########################################

def convert_nuscenes_to_coco_overlap(nusc, sensor_name, angles, overlap_threshold=0.1,
                                     output_json="overlap_coco_gt.json"):
    """
    Generate a COCO-style JSON for a given sensor (e.g., "CAM_FRONT" or "CAM_FRONT_LEFT"),
    keeping only bounding boxes within a specified angular range.
    Computes a 'completeness' score (clipped area / original projected area) and
    saves the 'orig_token' and 'sensor' for deduplication.
    """
    coco = {"images": [], "annotations": [], "categories": []}
    category_set = {}
    image_id = 0
    ann_id = 0

    for sample in nusc.sample:
        if sensor_name not in sample["data"]:
            continue
        img, sd = get_sample_data(nusc, sample["token"], sensor_name)
        W, H = img.width, img.height
        K = get_intrinsics(nusc, sd)
        u_min_crop, u_max_crop = crop_bounds(K, angles[0], angles[1], W)

        annotations_this_image = []
        for ann_token in sample["anns"]:
            original_bbox = project_3d_box_to_2d(nusc, sd, ann_token)
            if original_bbox is None:
                continue
            orig_u_min, orig_v_min, orig_u_max, orig_v_max = original_bbox
            original_area = (orig_u_max - orig_u_min) * (orig_v_max - orig_v_min)
            clipped = clip_box_to_image(original_bbox, W, H)
            if clipped is None:
                continue
            if horizontal_overlap_ratio(clipped, u_min_crop, u_max_crop) < overlap_threshold:
                continue
            clipped_area = (clipped[2] - clipped[0]) * (clipped[3] - clipped[1])
            completeness = clipped_area / original_area if original_area > 0 else 0.0

            ann_record = nusc.get("sample_annotation", ann_token)
            cat_name = ann_record["category_name"]
            if cat_name not in category_set:
                category_set[cat_name] = len(category_set) + 1
            cat_id = category_set[cat_name]

            x1, y1, x2, y2 = clipped
            w_box = x2 - x1
            h_box = y2 - y1

            annotations_this_image.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": cat_id,
                "bbox": [x1, y1, w_box, h_box],
                "area": w_box * h_box,
                "iscrowd": 0,
                "orig_token": ann_token,
                "sensor": sensor_name,
                "completeness": completeness
            })
            ann_id += 1

        if annotations_this_image:
            coco["images"].append({
                "id": image_id,
                "file_name": sd["filename"],
                "width": W,
                "height": H
            })
            coco["annotations"].extend(annotations_this_image)
            image_id += 1

    for cat_name, cat_id in category_set.items():
        coco["categories"].append({
            "id": cat_id,
            "name": cat_name
        })

    with open(output_json, "w") as f:
        json.dump(coco, f)
    print(f"{sensor_name} Overlap-region GT saved to {output_json}")
    return output_json

########################################
# (B1) Naive Merge (for comparison)
########################################

def merge_coco_json_naive(json_path1, json_path2, output_json):
    """
    Naively merge two COCO files by concatenating images and annotations.
    """
    with open(json_path1, 'r') as f:
        coco1 = json.load(f)
    with open(json_path2, 'r') as f:
        coco2 = json.load(f)
    merged = {}
    merged['categories'] = coco1['categories']
    merged['images'] = coco1['images'].copy()
    merged['annotations'] = coco1['annotations'].copy()
    image_offset = len(coco1['images'])
    ann_offset = len(coco1['annotations'])
    for img in coco2['images']:
        img['id'] += image_offset
    merged['images'].extend(coco2['images'])
    new_annotations = []
    for ann in coco2['annotations']:
        ann['image_id'] += image_offset
        ann['id'] += ann_offset
        new_annotations.append(ann)
    merged['annotations'].extend(new_annotations)
    with open(output_json, 'w') as f:
        json.dump(merged, f)
    print(f"Naive merged COCO GT saved to {output_json}")
    return output_json

########################################
# (B2) Partial Deduplication (Threshold setting)
#     For each group of duplicates (same orig_token), if the difference in completeness 
#     (max - min) is greater than delta_thresh, keep only the best (highest completeness);
#     otherwise, keep all.
########################################

def merge_coco_json_partial_deduplication(json_path1, json_path2, output_json, delta_thresh=0.1):
    """
    Merge two COCO files by grouping annotations with the same 'orig_token'.
    For each group, if the difference between the maximum and minimum completeness
    is greater than delta_thresh, keep only the annotation with the highest completeness.
    Otherwise, keep all annotations in that group.
    """
    with open(json_path1, 'r') as f:
        coco1 = json.load(f)
    with open(json_path2, 'r') as f:
        coco2 = json.load(f)
    merged = {}
    merged['categories'] = coco1['categories']
    merged['images'] = coco1['images'].copy()
    image_offset = len(coco1['images'])
    for img in coco2['images']:
        img['id'] += image_offset
    merged['images'].extend(coco2['images'])
    naive_annotations = []
    naive_annotations.extend(coco1['annotations'])
    ann_offset = len(coco1['annotations'])
    for ann in coco2['annotations']:
        ann['image_id'] += image_offset
        ann['id'] += ann_offset
        naive_annotations.append(ann)
    token_groups = {}
    for ann in naive_annotations:
        token = ann.get("orig_token")
        if token is None:
            token = f"no_token_{ann['id']}"
        token_groups.setdefault(token, []).append(ann)
    final_annotations = []
    for token, group in token_groups.items():
        if len(group) == 1:
            final_annotations.append(group[0])
        else:
            completeness_values = [ann.get("completeness", 0.0) for ann in group]
            max_comp = max(completeness_values)
            min_comp = min(completeness_values)
            if max_comp - min_comp > delta_thresh:
                best_ann = max(group, key=lambda a: a.get("completeness", 0.0))
                final_annotations.append(best_ann)
            else:
                final_annotations.extend(group)
    for i, ann in enumerate(final_annotations):
        ann['id'] = i
    merged['annotations'] = final_annotations
    with open(output_json, 'w') as f:
        json.dump(merged, f)
    print(f"Partially deduplicated COCO GT saved to {output_json}")
    return output_json

########################################
# (C) Convert COCO to YOLO Labels, Copy Images, and Print Stats
########################################

def coco_to_yolo(coco_json_path, images_folder, labels_folder):
    with open(coco_json_path, 'r') as f:
        data = json.load(f)
    images_info = {img['id']: img for img in data['images']}
    annotations = {}
    for ann in data['annotations']:
        annotations.setdefault(ann['image_id'], []).append(ann)
    os.makedirs(labels_folder, exist_ok=True)
    for image_id, img_info in images_info.items():
        width = img_info['width']
        height = img_info['height']
        base_name = osp.splitext(osp.basename(img_info['file_name']))[0]
        label_file = osp.join(labels_folder, base_name + '.txt')
        lines = []
        for ann in annotations.get(image_id, []):
            cat = ann['category_id'] - 1
            x, y, w, h = ann['bbox']
            x_center = (x + w/2.0) / width
            y_center = (y + h/2.0) / height
            lines.append(f"{cat} {x_center:.6f} {y_center:.6f} {w/width:.6f} {h/height:.6f}")
        with open(label_file, 'w') as f:
            f.write("\n".join(lines))
    print(f"Converted COCO annotations from {coco_json_path} into YOLO labels in {labels_folder}.")

def copy_overlap_images(coco_json_path, source_images_folder, dest_images_folder):
    with open(coco_json_path, 'r') as f:
        data = json.load(f)
    os.makedirs(dest_images_folder, exist_ok=True)
    copied = 0
    for img in data['images']:
        base_name = osp.basename(img['file_name'])
        src = osp.join(source_images_folder, base_name)
        dst = osp.join(dest_images_folder, base_name)
        if osp.exists(src):
            copy2(src, dst)
            copied += 1
        else:
            print(f"Warning: {src} not found.")
    print(f"Copied {copied} images to {dest_images_folder}.")

def create_data_yaml(coco_json_path, images_folder, output_yaml="data.yaml"):
    with open(coco_json_path, 'r') as f:
        data = json.load(f)
    nc = len(data['categories'])
    names = [cat['name'] for cat in data['categories']]
    content = f"""\
train: {images_folder}
val: {images_folder}
nc: {nc}
names: {names}
"""
    with open(output_yaml, "w") as f:
        f.write(content)
    print(f"data.yaml saved to {output_yaml}")

########################################
# (C2) Print COCO Stats with Correct Camera Distribution
########################################

def print_coco_stats(coco_json_path, dataset_name="Unknown"):
    with open(coco_json_path, "r") as f:
        data = json.load(f)
    images = data["images"]
    annotations = data["annotations"]
    categories = data["categories"]
    print(f"\n=== Dataset Stats: {dataset_name} ===")
    print(f"Number of images: {len(images)}")
    print(f"Number of annotations: {len(annotations)}")
    front_count_img = 0
    fl_count_img = 0
    image_id_to_camera = {}
    for img in images:
        file_name = img["file_name"]
        if "CAM_FRONT_LEFT" in file_name:
            fl_count_img += 1
            image_id_to_camera[img["id"]] = "front_left"
        elif "CAM_FRONT" in file_name:
            front_count_img += 1
            image_id_to_camera[img["id"]] = "front"
        else:
            image_id_to_camera[img["id"]] = "unknown"
    print(f"  # images from CAM_FRONT: {front_count_img}")
    print(f"  # images from CAM_FRONT_LEFT: {fl_count_img}")
    front_count_ann = 0
    fl_count_ann = 0
    for ann in annotations:
        img_id = ann["image_id"]
        cam = image_id_to_camera.get(img_id, "unknown")
        if cam == "front":
            front_count_ann += 1
        elif cam == "front_left":
            fl_count_ann += 1
    print(f"  # annotations from CAM_FRONT images: {front_count_ann}")
    print(f"  # annotations from CAM_FRONT_LEFT images: {fl_count_ann}")
    cat_id_to_name = {cat["id"]: cat["name"] for cat in categories}
    cat_count = {}
    for ann in annotations:
        c_id = ann["category_id"]
        cat_count[c_id] = cat_count.get(c_id, 0) + 1
    print("  Category distribution:")
    for c_id, count in sorted(cat_count.items()):
        cat_name = cat_id_to_name.get(c_id, "unknown")
        print(f"    {cat_name} (id={c_id}): {count}")

########################################
# (D) Detection + Evaluation Functions
########################################

def get_yolo_detections(yolo_model, img, u_min_offset=0):
    results = yolo_model(img)
    dets = []
    for det in results[0].boxes.data.cpu().numpy():
        x1, y1, x2, y2, score, cls = det
        x1 += u_min_offset
        x2 += u_min_offset
        dets.append({
            "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
            "score": float(score),
            "category_id": int(cls) + 1
        })
    return dets

def filter_detections_to_overlap(detections, u_min_crop, u_max_crop, threshold=0.1):
    filtered = []
    for det in detections:
        x, y, w, h = det["bbox"]
        if horizontal_overlap_ratio((x, y, x+w, y+h), u_min_crop, u_max_crop) >= threshold:
            filtered.append(det)
    return filtered

def generate_detections_reduced(nusc, sample_tokens, sensor_name, angles, yolo_model, output_json):
    detections_all = []
    for i, token in enumerate(sample_tokens):
        img, sd = get_sample_data(nusc, token, sensor_name)
        W = img.width
        K = get_intrinsics(nusc, sd)
        u_min_crop, u_max_crop = crop_bounds(K, angles[0], angles[1], W)
        img_np = np.array(img)
        img_bgr = img_np[:, :, ::-1]
        dets = get_yolo_detections(yolo_model, img_bgr, u_min_offset=0)
        filtered = filter_detections_to_overlap(dets, u_min_crop, u_max_crop, threshold=0.1)
        for det in filtered:
            det["image_id"] = i
        detections_all.extend(filtered)
    with open(output_json, "w") as f:
        json.dump(detections_all, f)
    print(f"Reduced detections saved to {output_json}")
    return output_json

def evaluate_coco(coco_gt_path, coco_dt_path, iou_thrs=None):
    coco_gt = COCO(coco_gt_path)
    coco_dt = coco_gt.loadRes(coco_dt_path)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    if iou_thrs is not None:
        coco_eval.params.iouThrs = iou_thrs
    coco_eval.evaluate()
    coco_eval.accumulate()
    print(f"\n=== Evaluation for {coco_dt_path} ===")
    coco_eval.summarize()
    return coco_eval

########################################
# (E) Main Pipeline for Pair 2:
#     Sensors: CAM_FRONT and CAM_FRONT_LEFT with specified angular ranges.
#     For CAM_FRONT, use angles (-35, -20).
#     For CAM_FRONT_LEFT, use angles (20, 35).
#     Use partial deduplication with a delta threshold.
########################################

if __name__ == "__main__":
    # 0. Initialize nuScenes
    nusc = NuScenes(version='v1.0-mini',
                    dataroot='',
                    verbose=True)

    # ------------------------------------------------------------
    # 1. Generate Per-Camera COCO GT with completeness
    # ------------------------------------------------------------
    gt_front = "overlap_coco_gt_front.json"
    # For CAM_FRONT, use angular range (-35, -20)
    convert_nuscenes_to_coco_overlap(nusc, sensor_name="CAM_FRONT",
                                     angles=(-35, -20),
                                     output_json=gt_front)

    gt_front_left = "overlap_coco_gt_front_left.json"
    # For CAM_FRONT_LEFT, use angular range (20, 35)
    convert_nuscenes_to_coco_overlap(nusc, sensor_name="CAM_FRONT_LEFT",
                                     angles=(20, 35),
                                     output_json=gt_front_left)

    # Print stats for each
    print_coco_stats(gt_front, dataset_name="Front GT")
    print_coco_stats(gt_front_left, dataset_name="Front Left GT")

    # ------------------------------------------------------------
    # 2. Create Merged Datasets:
    #    (a) Naive Merge (for evaluation)
    #    (b) Partial Deduplication with delta_thresh
    # ------------------------------------------------------------
    naive_merged_gt = "overlap_coco_gt_merged_naive.json"
    merge_coco_json_naive(gt_front, gt_front_left, naive_merged_gt)
    print_coco_stats(naive_merged_gt, dataset_name="Naive Merged GT")

    partial_dedup_gt = "overlap_coco_gt_merged_partial.json"
    merge_coco_json_partial_deduplication(gt_front, gt_front_left, partial_dedup_gt, delta_thresh=1.0)
    print_coco_stats(partial_dedup_gt, dataset_name="Partial Deduplicated Merged GT")

    # ------------------------------------------------------------
    # 3. Copy images (from both CAM_FRONT and CAM_FRONT_LEFT) into one folder
    # ------------------------------------------------------------
    dest_images_folder = ""
    source_images_folder_front = ""
    source_images_folder_front_left = ""
    copy_overlap_images(gt_front, source_images_folder_front, dest_images_folder)
    copy_overlap_images(gt_front_left, source_images_folder_front_left, dest_images_folder)

    # ------------------------------------------------------------
    # 4. Train on the PARTIALLY deduplicated merged dataset
    # ------------------------------------------------------------
    labels_folder = ""
    coco_to_yolo(partial_dedup_gt, dest_images_folder, labels_folder)
    create_data_yaml(partial_dedup_gt, dest_images_folder, output_yaml="data.yaml")

    model = YOLO("yolov8n.pt")
    model.train(data="data.yaml", epochs=50, imgsz=640, device=device)
    train_val_results = model.val(data="data.yaml", device=device)
    print("Validation results on partially deduplicated merged dataset:", train_val_results)

    # ------------------------------------------------------------
    # 5. Evaluation:
    #    (a) On CAM_FRONT subset
    #    (b) On CAM_FRONT_LEFT subset
    #    (c) On FULL (Naive) Merged dataset
    # ------------------------------------------------------------
    #####################
    # 5(a) Evaluate on CAM_FRONT
    #####################
    sample_tokens_front = []
    with open(gt_front, 'r') as f:
        gt_data_front = json.load(f)
    gt_files_front = set(img['file_name'] for img in gt_data_front['images'])
    for sample in nusc.sample:
        if "CAM_FRONT" in sample["data"]:
            sd = nusc.get('sample_data', sample["data"]["CAM_FRONT"])
            if sd["filename"] in gt_files_front:
                sample_tokens_front.append(sample["token"])
    detection_file_front = "detections_CAM_FRONT.json"
    generate_detections_reduced(nusc, sample_tokens_front,
                                sensor_name="CAM_FRONT",
                                angles=(-35, -20),
                                yolo_model=model,
                                output_json=detection_file_front)
    evaluate_coco(gt_front, detection_file_front, iou_thrs=np.array([0.5]))

    #####################
    # 5(b) Evaluate on CAM_FRONT_LEFT
    #####################
    sample_tokens_front_left = []
    with open(gt_front_left, 'r') as f:
        gt_data_front_left = json.load(f)
    gt_files_front_left = set(img['file_name'] for img in gt_data_front_left['images'])
    for sample in nusc.sample:
        if "CAM_FRONT_LEFT" in sample["data"]:
            sd = nusc.get('sample_data', sample["data"]["CAM_FRONT_LEFT"])
            if sd["filename"] in gt_files_front_left:
                sample_tokens_front_left.append(sample["token"])
    detection_file_front_left = "detections_CAM_FRONT_LEFT.json"
    generate_detections_reduced(nusc, sample_tokens_front_left,
                                sensor_name="CAM_FRONT_LEFT",
                                angles=(20, 35),
                                yolo_model=model,
                                output_json=detection_file_front_left)
    evaluate_coco(gt_front_left, detection_file_front_left, iou_thrs=np.array([0.5]))

    #####################
    # 5(c) Evaluate on FULL (Naive) Merged dataset
    #####################
    detection_file_front_naive = "detections_CAM_FRONT_naive_eval.json"
    generate_detections_reduced(nusc, sample_tokens_front,
                                sensor_name="CAM_FRONT",
                                angles=(-35, -20),
                                yolo_model=model,
                                output_json=detection_file_front_naive)
    detection_file_front_left_naive = "detections_CAM_FRONT_LEFT_naive_eval.json"
    generate_detections_reduced(nusc, sample_tokens_front_left,
                                sensor_name="CAM_FRONT_LEFT",
                                angles=(20, 35),
                                yolo_model=model,
                                output_json=detection_file_front_left_naive)

    with open(detection_file_front_naive, 'r') as f:
        dets_front_naive = json.load(f)
    with open(detection_file_front_left_naive, 'r') as f:
        dets_front_left_naive = json.load(f)

    with open(gt_front, 'r') as f:
        front_data_only = json.load(f)
    image_offset = len(front_data_only['images'])

    for det in dets_front_left_naive:
        det["image_id"] += image_offset

    merged_detections_naive_eval = dets_front_naive + dets_front_left_naive
    detection_file_naive_merged_eval = "detections_naive_merged_eval.json"
    with open(detection_file_naive_merged_eval, 'w') as f:
        json.dump(merged_detections_naive_eval, f)
    print(f"Merged naive detection results saved to {detection_file_naive_merged_eval}")

    print("Evaluating model (trained on partially deduplicated set) on the FULL naive merged dataset:")
    evaluate_coco(naive_merged_gt, detection_file_naive_merged_eval, iou_thrs=np.array([0.5]))

Using GPU: NVIDIA GeForce RTX 4090
Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.289 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.
CAM_FRONT Overlap-region GT saved to overlap_coco_gt_front.json
CAM_FRONT_LEFT Overlap-region GT saved to overlap_coco_gt_front_left.json

=== Dataset Stats: Front GT ===
Number of images: 287
Number of annotations: 909
  # images from CAM_FRONT: 287
  # images from CAM_FRONT_LEFT: 0
  # annotations from CAM_FRONT images: 909
  # annotations from CAM_FRONT_LEFT images: 0
  Category distribution:
    vehicle.truck (id=1): 71
    human.pedestrian.adult (id=2): 226
    movable_object.pushable_pullable (id=3): 7
    vehicle.motorcycle (id=4): 33
    movable_object.trafficcone (id=5): 110
    movable_object.barrier (id=6): 35
    vehicle.c

train: Scanning C:\pair2\overlap_dataset\labels... 567 images, 0 backgrounds, 0 corrupt: 100%|██████████| 567/567 [00:00<00:00, 3382.06it/s]


train: New cache created: C:\pair2\overlap_dataset\labels.cache


val: Scanning C:\pair2\overlap_dataset\labels.cache... 567 images, 0 backgrounds, 0 corrupt: 100%|██████████| 567/567 [00:00<?, ?it/s]


Plotting labels to runs\detect\train108\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.0005, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\detect\train108
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        567       1693      0.756       0.68      0.729      0.565

50 epochs completed in 0.069 hours.
Optimizer stripped from runs\detect\train108\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train108\weights\best.pt, 6.2MB

Validating runs\detect\train108\weights\best.pt...
Ultralytics 8.3.94  Python-3.9.13 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 72 layers, 3,008,768 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:02<00:00,  6.74it/s]


                   all        567       1693      0.756       0.68      0.725      0.565
         vehicle.truck        114        134      0.928      0.873      0.911      0.802
human.pedestrian.adult        202        438      0.768      0.521      0.639      0.429
movable_object.pushable_pullable         14         14          1      0.996      0.995      0.861
    vehicle.motorcycle         36         56      0.805      0.821      0.855      0.534
movable_object.trafficcone         61        200      0.694      0.475      0.548      0.305
movable_object.barrier         23         51      0.624      0.765      0.797      0.527
           vehicle.car        352        622       0.86      0.887       0.93       0.74
       vehicle.bicycle         15         19       0.79      0.737      0.869      0.658
human.pedestrian.child          7          7          1          0    0.00729    0.00353
static_object.bicycle_rack         14         14      0.695          1      0.995       0.83
   

val: Scanning C:\pair2\overlap_dataset\labels.cache... 567 images, 0 backgrounds, 0 corrupt: 100%|██████████| 567/567 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 36/36 [00:02<00:00, 13.65it/s]


                   all        567       1693      0.755       0.68      0.729      0.568
         vehicle.truck        114        134      0.935      0.873      0.911      0.802
human.pedestrian.adult        202        438      0.779      0.522      0.644      0.431
movable_object.pushable_pullable         14         14          1      0.996      0.995      0.866
    vehicle.motorcycle         36         56      0.806      0.821      0.855      0.541
movable_object.trafficcone         61        200      0.695      0.475      0.549      0.306
movable_object.barrier         23         51      0.624      0.765      0.797      0.525
           vehicle.car        352        622       0.86      0.886      0.929      0.741
       vehicle.bicycle         15         19      0.748      0.737      0.863      0.661
human.pedestrian.child          7          7          1          0    0.00729    0.00353
static_object.bicycle_rack         14         14      0.695          1      0.995       0.84
   